In [1]:
from pathlib import Path
from conf.behavior_cloning.diffusion.five_demos.default import config
from tapas_gmm.dataset.scene import SceneDataset
from tapas_gmm.dataset.bc import BCDataset
from tapas_gmm.encoder.encoder import ObservationEncoderConfig
from tapas_gmm.policy.diffusion import DiffusionPolicy, DiffusionPolicyTrainingConfig
from tapas_gmm.behavior_cloning import run_training

2026-06-23 15:33:59.509 | INFO     |  Running on cpu


In [2]:
data_root = Path("../outputs/bimanual_dataset")
horizon = 16
n_obs_steps = 2
n_action_steps = 8
epochs = 20

In [3]:
config.policy.action_dim = 16
config.policy.obs_dim = 35
config.policy.horizon = horizon
config.policy.n_obs_steps = n_obs_steps
config.policy.n_action_steps = n_action_steps
config.policy.training = DiffusionPolicyTrainingConfig(lr_num_epochs=epochs)
config.policy.obs_encoder = ObservationEncoderConfig(
    ee_pose=True,
    object_poses=True,
)

config.policy.unet.input_dim = 16
config.policy.unet.global_cond_dim = 35 * n_obs_steps
config.policy.unet.down_dims = (64, 128, 256)

In [4]:
config.bc_data.fragment_length = horizon + 1
config.bc_data.pre_padding = n_obs_steps - 1
config.bc_data.post_padding = n_action_steps - 1
config.bc_data.cameras = tuple()

In [5]:
config.training.epochs = epochs
config.training.eval_freq = 5

In [6]:
config.data_loader.batch_size = 8
config.data_loader.eval_batchsize = 8


In [7]:
loaded_dataset = SceneDataset(data_root=data_root)

bc_dataset = BCDataset(
    scene_dataset=loaded_dataset,
    config=config.bc_data,
)

policy = DiffusionPolicy(config.policy)

2026-06-23 15:34:05.667 | INFO     |  Initializing datasete using ../outputs/bimanual_dataset/metadata.json
2026-06-23 15:34:05.790 | INFO     |  Extracted gt object labels []
2026-06-23 15:34:05.790 | INFO     |  Extracted tsdf object labels []
2026-06-23 15:34:05.790 | INFO     |  Initializing BCDataset:
2026-06-23 15:34:05.790 | INFO     |    Training on fragments of length 17.
2026-06-23 15:34:05.790 | INFO     |    Loading raw data for encoder.
2026-06-23 15:34:05.790 | INFO     |  Initializing DiffusionPolicy:
2026-06-23 15:34:05.791 | INFO     |    Initializing Policy:
2026-06-23 15:34:06.007 | INFO     |    number of parameters: 5525968
2026-06-23 15:34:06.127 | INFO     |    No encoder config provided. Using None.
None


In [8]:
import wandb
wandb.init(mode="disabled")

run_training(
    policy,
    bc_dataset,
    config,
    "../outputs/bimanual_diffusion_policy",
)

2026-06-23 15:34:06.437 | INFO     |  No datasplit specified.
2026-06-23 15:34:06.537 | INFO     |    Setting action scaling for optimal policy performance. Using DP normalizer implementation.
2026-06-23 15:34:08.545 | INFO     |  Beginning training.


  0%|          | 0/20 [00:00<?, ?it/s]

[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.


In [9]:
policy.to_disk("../outputs/bimanual_diffusion_policy.pt")

2026-06-23 16:00:04.048 | INFO     |  Saving policy at ../outputs/bimanual_diffusion_policy.pt
